# Mission Eagle-1 — DQN vs PPO : quel algorithme pour notre pilote ?

PPO nous donne de bons resultats, mais le brief de mission recommandait **DQN** pour les espaces d'action discrets. Est-ce que c'est vraiment mieux ? Il n'y a qu'une facon de savoir : tester.

## DQN en bref

On a deja vu le **Q-Learning** sur FrozenLake (notebook Q-Learning). DQN, c'est la meme idee, mais au lieu d'une Q-table, on utilise un **reseau de neurones** pour estimer Q(s, a).

|                       | Q-Learning classique            | DQN                                  | PPO                  |
| --------------------- | ------------------------------- | ------------------------------------ | -------------------- |
| **Stockage**          | Q-table (tableau)               | Reseau de neurones                   | Reseau de neurones   |
| **Espaces d'etats**   | Discrets uniquement             | Discrets ou continus                 | Discrets ou continus |
| **Espaces d'actions** | Discrets uniquement             | Discrets uniquement                  | Discrets ET continus |
| **Type**              | Off-policy                      | Off-policy                           | On-policy            |
| **Replay buffer**     | Non                             | Oui (stocke les experiences passees) | Non                  |
| **Stabilite**         | Tres stable (converge toujours) | Peut etre instable                   | Tres stable          |

> **Note methodo** : "Off-policy" signifie que l'agent peut apprendre a partir d'experiences passees (stockees dans un replay buffer). "On-policy" signifie qu'il apprend uniquement de ses experiences les plus recentes. L'off-policy est plus sample-efficient en theorie, mais l'on-policy est souvent plus stable en pratique.


In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.evaluation import evaluate_policy
import time

## 1. DQN avec parametres par defaut

Commencons par un DQN "out of the box", comme on a fait pour PPO.


In [2]:
env = gym.make("LunarLander-v3")

print("Entrainement DQN — 300k timesteps (parametres par defaut)")
print("=" * 55)

model_dqn = DQN("MlpPolicy", env, verbose=1, tensorboard_log="./logs/dqn")
model_dqn.learn(total_timesteps=300_000, tb_log_name="dqn_default")
env.close()

eval_env = gym.make("LunarLander-v3")
mean_reward, std_reward = evaluate_policy(model_dqn, eval_env, n_eval_episodes=50)
eval_env.close()

print(f"\nDQN par defaut (300k steps) :")
print(f"  Reward moyen : {mean_reward:.1f} +/- {std_reward:.1f}")

/Users/ppluton/dev/tuto_baseline_rl/.venv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Entrainement DQN — 300k timesteps (parametres par defaut)
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./logs/dqn/dqn_default_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 89.2     |
|    ep_rew_mean      | -219     |
|    exploration_rate | 0.989    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 3805     |
|    time_elapsed     | 0        |
|    total_timesteps  | 357      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 1.51     |
|    n_updates        | 64       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 91.2     |
|    ep_rew_mean      | -187     |
|    exploration_rate | 0.977    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 5025     |
|    ti

/Users/ppluton/dev/tuto_baseline_rl/.venv/lib/python3.11/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(



DQN par defaut (300k steps) :
  Reward moyen : -53.4 +/- 32.0


## 2. Optimisation rapide de DQN

On fait un tour rapide de tuning pour donner sa chance a DQN. Les hyperparametres cles de DQN sont differents de PPO :

| Hyperparametre          | Defaut SB3 | Role                                                            |
| ----------------------- | ---------- | --------------------------------------------------------------- |
| `learning_rate`         | 1e-4       | Vitesse d'apprentissage                                         |
| `buffer_size`           | 1000000    | Taille du replay buffer (memoire des experiences passees)       |
| `exploration_fraction`  | 0.1        | Fraction du training consacree a l'exploration (epsilon-greedy) |
| `exploration_final_eps` | 0.05       | Epsilon final (probabilite d'action aleatoire)                  |
| `batch_size`            | 32         | Taille des batchs echantillonnes du replay buffer               |

On va tester quelques combinaisons raisonnables.


In [3]:
configs = [
    {"name": "lr_high", "learning_rate": 5e-4, "exploration_fraction": 0.2},
    {"name": "lr_low_explore", "learning_rate": 1e-4, "exploration_fraction": 0.3},
    {
        "name": "big_batch",
        "learning_rate": 3e-4,
        "batch_size": 64,
        "exploration_fraction": 0.15,
    },
]

results_dqn = []

for config in configs:
    name = config.pop("name")
    print(f"\n{'=' * 55}")
    print(f"DQN config: {name} — {config}")
    print(f"{'=' * 55}")

    env = gym.make("LunarLander-v3")
    model = DQN("MlpPolicy", env, verbose=0, tensorboard_log="./logs/dqn", **config)

    start = time.time()
    model.learn(total_timesteps=300_000, tb_log_name=f"dqn_{name}")
    duration = time.time() - start

    eval_env = gym.make("LunarLander-v3")
    mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=50)
    eval_env.close()
    env.close()

    results_dqn.append(
        {
            "name": name,
            "mean_reward": mean_reward,
            "std_reward": std_reward,
            "duration": duration,
        }
    )

    print(f"  Reward: {mean_reward:.1f} +/- {std_reward:.1f} ({duration:.0f}s)")

print("\n\nRecap DQN :")
print(f"{'Config':>20} | {'Reward':>10} | {'Std':>8} | {'Temps':>6}")
print("-" * 50)
for r in results_dqn:
    print(
        f"{r['name']:>20} | {r['mean_reward']:>10.1f} | {r['std_reward']:>8.1f} | {r['duration']:>5.0f}s"
    )


DQN config: lr_high — {'learning_rate': 0.0005, 'exploration_fraction': 0.2}
  Reward: -29.2 +/- 66.6 (53s)

DQN config: lr_low_explore — {'learning_rate': 0.0001, 'exploration_fraction': 0.3}
  Reward: -15.9 +/- 154.2 (62s)

DQN config: big_batch — {'learning_rate': 0.0003, 'batch_size': 64, 'exploration_fraction': 0.15}
  Reward: -47.2 +/- 96.2 (66s)


Recap DQN :
              Config |     Reward |      Std |  Temps
--------------------------------------------------
             lr_high |      -29.2 |     66.6 |    53s
      lr_low_explore |      -15.9 |    154.2 |    62s
           big_batch |      -47.2 |     96.2 |    66s


## 3. Le verdict : PPO vs DQN

Chargeons notre meilleur PPO (du notebook 02) et comparons-le au meilleur DQN qu'on vient d'entrainer.


In [4]:
# Charger le meilleur PPO
model_ppo = PPO.load("models/ppo_optimized")

eval_env = gym.make("LunarLander-v3")

# Evaluer PPO sur 100 episodes
ppo_mean, ppo_std = evaluate_policy(model_ppo, eval_env, n_eval_episodes=100)

# Prendre le meilleur DQN et l'evaluer aussi sur 100
best_dqn_idx = np.argmax([r["mean_reward"] for r in results_dqn])
best_dqn_name = results_dqn[best_dqn_idx]["name"]

# Re-entrainer le meilleur DQN (on n'a pas sauvegarde)
best_config = configs[best_dqn_idx]
env = gym.make("LunarLander-v3")
model_dqn_best = DQN("MlpPolicy", env, verbose=0, **best_config)
model_dqn_best.learn(total_timesteps=300_000)
env.close()

dqn_mean, dqn_std = evaluate_policy(model_dqn_best, eval_env, n_eval_episodes=100)
eval_env.close()

print("=" * 55)
print("COMPARAISON FINALE (100 episodes)")
print("=" * 55)
print(f"{'Algorithme':>12} | {'Reward':>10} | {'Std':>8}")
print("-" * 36)
print(f"{'PPO':>12} | {ppo_mean:>10.1f} | {ppo_std:>8.1f}")
print(f"{'DQN':>12} | {dqn_mean:>10.1f} | {dqn_std:>8.1f}")
print()

winner = "PPO" if ppo_mean > dqn_mean else "DQN"
print(f"Vainqueur : {winner}")

COMPARAISON FINALE (100 episodes)
  Algorithme |     Reward |      Std
------------------------------------
         PPO |      243.7 |     51.3
         DQN |      115.0 |    104.4

Vainqueur : PPO


## Conclusion

Les resultats sont sans appel :

- **PPO optimise** : 243.7 +/- 51.3 sur 100 episodes
- **Meilleur DQN** (lr=1e-4, exploration_fraction=0.3) : 115.0 +/- 104.4 sur 100 episodes apres re-entrainement

**PPO ecrase DQN** sur LunarLander-v3. DQN n'arrive meme pas a depasser 150 en moyenne, alors que PPO est confortablement au-dessus de 200. En plus, l'ecart-type de DQN (104.4) est deux fois celui de PPO (51.3) — non seulement il est moins bon, mais il est aussi beaucoup moins stable.

Pourquoi ? Quelques hypotheses :

- DQN est plus sensible aux hyperparametres et demande plus de tuning fin
- Le replay buffer de DQN peut etre un avantage en theorie, mais PPO est simplement plus stable en pratique
- 300k timesteps est peut-etre insuffisant pour DQN (il est connu pour converger plus lentement)

> **Note methodo** : En ML, les regles generales ("DQN pour discret, SAC pour continu") sont des points de depart, pas des verites absolues. Toujours tester soi-meme sur son probleme specifique. Ici, PPO gagne haut la main.

On garde **PPO** comme pilote automatique pour Eagle-1. C'etait le bon choix.

**Prochaine etape** : evaluation finale et generation de video ->
